# Python Coding Practices for AI Applications

Habits that keep AI code readable, safe and easy to change.

**You will learn to:**

- Keep secrets out of code
- Structure a script with small, typed functions
- Use constants and dataclasses for configuration
- Log instead of printing
- Validate inputs and test with simple checks

## 1. Project layout

```text
my-ai-app/
├── .venv/               # virtual environment (not committed)
├── .env                 # secrets (not committed)
├── .gitignore
├── requirements.txt
├── README.md
├── src/
│   └── my_ai_app/
│       ├── __init__.py
│       ├── config.py    # settings
│       ├── client.py    # model calls
│       └── prompts.py   # prompt templates
└── tests/
    └── test_prompts.py
```

Keep one job per file. Keep notebooks for exploration and move stable code into modules.

## 2. Keep secrets out of code

Never type an API key into a script or notebook. Read it from an **environment variable**.
Locally, a `.env` file loaded by `python-dotenv` fills those variables. Add `.env` to `.gitignore`.

In [ ]:
import os


def require_env(name: str) -> str:
    """Return the value of environment variable `name` or fail with a clear message."""
    value = os.environ.get(name)
    if not value:
        raise RuntimeError(f"Set the {name} environment variable before running.")
    return value


# Optional setting with a safe default
log_level = os.environ.get("APP_LOG_LEVEL", "INFO")
print("Log level:", log_level)

try:
    require_env("A_SECRET_THAT_IS_NOT_SET")
except RuntimeError as error:
    print("Error:", error)

In [ ]:
# Typical local pattern (needs: pip install python-dotenv, and a .env file)
try:
    from dotenv import load_dotenv
    load_dotenv()                     # copies values from .env into os.environ
    print("dotenv loaded")
except ImportError:
    print("python-dotenv is not installed; skipping")

> **Warning:** never print a secret, log it, or paste it into a prompt. If you commit one by accident, revoke it and create a new one.

## 3. Names, constants and configuration

In [ ]:
from dataclasses import dataclass

DEFAULT_MODEL = "your-model-id"          # constants: UPPER_SNAKE_CASE
DEFAULT_MAX_TOKENS = 1024


@dataclass(frozen=True)                  # frozen: cannot be changed after creation
class Config:
    model: str = DEFAULT_MODEL
    max_tokens: int = DEFAULT_MAX_TOKENS
    temperature: float = 0.7

    def __post_init__(self):
        if not 0 <= self.temperature <= 1:
            raise ValueError("temperature must be between 0 and 1")


config = Config(temperature=0.2)
print(config)

Avoid "magic numbers" and repeated strings. Give them a name once.

## 4. Small functions with type hints

In [ ]:
def build_prompt(question: str, context: str = "") -> str:
    """Return a prompt that includes optional context."""
    parts = []
    if context:
        parts.append(f"Context:\n{context}")
    parts.append(f"Question: {question}")
    return "\n\n".join(parts)


def count_words(text: str) -> int:
    """Return the number of words in `text`."""
    return len(text.split())


prompt = build_prompt("What is RAG?", context="RAG retrieves documents before answering.")
print(prompt)
print(count_words(prompt), "words")

Guidelines:

- One function, one job. If you need "and" to describe it, split it.
- Return values instead of printing inside functions, so they are easy to test.
- Avoid changing global variables or the arguments you were given.

## 5. Log instead of print

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
logger = logging.getLogger("my_ai_app")

logger.debug("hidden at INFO level")
logger.info("Calling model with %d tokens of context", 1200)     # lazy formatting
logger.warning("Slow response: %.1fs", 4.2)

try:
    1 / 0
except ZeroDivisionError:
    logger.exception("Something failed")       # includes the traceback

Log what happened (request size, timing, outcome). Do **not** log secrets or private user data.

## 6. Validate inputs early

In [ ]:
def summarize_request(text: str, max_words: int = 100) -> dict:
    """Validate a request and return the data to send."""
    if not isinstance(text, str) or not text.strip():
        raise ValueError("text must be a non-empty string")
    if not 1 <= max_words <= 1000:
        raise ValueError("max_words must be between 1 and 1000")
    return {"text": text.strip(), "max_words": max_words}


for args in [("Hello world", 50), ("   ",), ("ok", 5000)]:
    try:
        print(summarize_request(*args))
    except ValueError as error:
        print("Rejected:", error)

Treat everything that comes from a user, a file or a model as **untrusted**. Check it before use.

## 7. Test with simple checks

Write a quick `assert` for each behaviour. Later, move them into `pytest` test files.

In [ ]:
assert count_words("one two three") == 3
assert count_words("") == 0
assert "Context:" in build_prompt("q", context="c")
assert "Context:" not in build_prompt("q")
print("all checks passed")

```python
# tests/test_prompts.py  (run with: pytest)
from my_ai_app.prompts import build_prompt

def test_build_prompt_includes_question():
    assert "Question: Hi" in build_prompt("Hi")
```

## 8. Style and tools

- Follow **PEP 8**: 4-space indents, `snake_case` names, lines under about 88 characters.
- Let a formatter do the work: `ruff format .` or `black .`. Let a linter catch mistakes: `ruff check .`.
- Write docstrings for modules, classes and functions. See the course documentation style guide.
- Pin versions in `requirements.txt` and recreate your environment from it.
- Use `pathlib.Path` instead of building paths with strings.
- Use `if __name__ == "__main__":` so files work as both scripts and modules.

### Checklist

- [ ] No secrets in code, notebooks or logs
- [ ] Settings in one place (constants or a config object)
- [ ] Functions are small, typed and documented
- [ ] Inputs validated; errors are specific and helpful
- [ ] Temporary failures are retried; permanent ones are not
- [ ] Logging instead of `print` outside notebooks
- [ ] A few automated checks run before each change

## Try it

1. Add a `Config` field `system_prompt` and make it reject an empty string.
2. Write `load_config_from_env()` that builds a `Config` from `APP_MODEL` and `APP_TEMPERATURE`, with defaults.
3. Replace any `print` calls in your own earlier code with `logger.info`.

In [ ]:
# Your answers here